In [0]:
# Load ecommerce transaction data from the source table into a Spark DataFrame
# This table contains all transaction records
events = spark.read.table("workspace.default.ecommerce_transactions")

In [0]:
# Save the loaded DataFrame as a Delta table for better performance and reliability
# Overwrite mode ensures the table is replaced if it already exists
events.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.events_table")

In [0]:
# Display detailed information about the newly created Delta table
# This includes metadata like number of rows, size, and creation time
display(
    spark.sql(
        """
        DESCRIBE DETAIL workspace.default.events_table
        """
    )
)

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,454923c0-fe5f-4fe2-9637-d0a75f3e6181,workspace.default.events_table,null,,2026-02-15T13:48:32.328Z,2026-02-15T13:48:45.000Z,List(),List(),1,460193,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
-- Create a new Delta table from the existing events_table using SQL
-- This ensures the data is available in another table for further analysis
Create table if not exists workspace.default.events_delta
using delta
as
SELECT * FROM workspace.default.events_table;

num_affected_rows,num_inserted_rows


In [0]:
# Try to write a DataFrame with a wrong schema to the Delta table
# This should fail and demonstrate schema enforcement in Delta tables
try:
    wrong_schema = spark.createDataFrame(
        [("a","b","c")],
        ["x","y","z"]
    )
    wrong_schema.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("workspace.default.events_table")
        
except Exception as e:
    print("Schema enforcement worked")
    print("Schema mismatch: The columns ['x','y','z'] do not match the schema of workspace.default.events_table.")
    print(e)

Schema enforcement worked
Schema mismatch: The columns ['x','y','z'] do not match the schema of workspace.default.events_table.
[_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 454923c0-fe5f-4fe2-9637-d0a75f3e6181).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- Transaction_ID: long (nullable = true)
-- User_Name: string (nullable = true)
-- Age: long (nullable = true)
-- Country: string (nullable = true)
-- Product_Category: string (nullable = true)
-- Purchase_Amount: double (nullable = true)
-- Payment_Method: string (nullable = true)
-- Transaction_Date: date (nullable = true)


Data schema:
root
-- x: string (nullable = true)
-- y: string (nullable = true)
-- z: string 

In [0]:
# Remove duplicate rows based on Transaction_ID to ensure each transaction is unique
removedup = events.dropDuplicates(["Transaction_ID"])

# Overwrite the Delta table with the deduplicated data
removedup.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.events_table")

# Display updated table details to confirm changes
display(
    spark.sql(
        """
        DESCRIBE DETAIL workspace.default.events_table
        """
    )
)

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,454923c0-fe5f-4fe2-9637-d0a75f3e6181,workspace.default.events_table,null,,2026-02-15T13:48:32.328Z,2026-02-15T14:15:24.000Z,List(),List(),1,453352,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
-- Find Transaction_IDs that appear more than once in the table
-- This helps identify any remaining duplicates after deduplication
select Transaction_ID, Count(*)
from workspace.default.events_table
group by Transaction_ID
Having Count(*) > 1;

Transaction_ID,Count(*)
